In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root       8854 F...m python3
/dev/nvidiactl:      root       8854 F...m python3
/dev/nvidia-uvm:     root       8854 F...m python3


In [2]:
# Check the GPU status
!nvidia-smi

Sun Jul 19 22:07:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             29W /   70W |       0MiB /  15360MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip install evaluate

In [4]:
import json
import torch
import evaluate
import pandas as pd
from datetime import datetime
from pathlib import Path
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    DataCollatorWithPadding, 
    Trainer, 
    TrainingArguments,
)
from datasets import load_dataset, Dataset

# Configurations

In [5]:
# Run configuration
SEED = 42
LANGUAGES = ['en', 'ar', 'de', 'el', 'es', 'hi', 'ru', 'th', 'tr', 'vi', 'zh']

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-1K-LoRA-Merged-v260623145250'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-1K-LoRA-Addition-v260713162325'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260712125532'
# ----
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719003350'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Averaging-v260719002351'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719130212'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Averaging-v260719131039'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719214552'
# ----
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LegameX-LoRA-Addition-v260718220644'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LegameX-LoRA-Averaging-v260718221730'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719123330'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-vi-5K-LoRA-Addition-v260719215134'
# ---
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438'

# Data configuration
# TEST_SIZE = 125
TEST_SIZE = 625
DATA_ID = 'google/xquad'
DATA_DIR = 'xquad.{lang}'
DATA_SPLIT = 'validation'

# Evaluation configuration
BATCH_SIZE = 16

# Set up the evaluation directory
eval_dir = f'./eval/xquad_xlmr/{TEST_SIZE}/{MODEL_ID}'
print(f"Evaluation directory: {eval_dir}")

Evaluation directory: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438


# Utilities

In [6]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size,
    data_id=DATA_ID,
    data_dir=DATA_DIR,
    data_split=DATA_SPLIT,
):
    assert '{lang}' in data_dir, "Data directory must contain a '{lang}' placeholder."
    
    dataset_stream = load_dataset(
        data_id,
        data_dir=data_dir.format(lang=lang),
        split=data_split,
        streaming=True,
    )

    test_data = []
    for i, example in enumerate(dataset_stream):
        if i < size:
            test_data.append(example)
        else:
            break

    return Dataset.from_list(test_data)

# Model

In [7]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')
model.eval()

print("device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

device: cuda:0


# Data

In [8]:
# Preprocess the test dataset for evaluation
def preprocess_squad(examples):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=384,
        return_offsets_mapping=True,
    )
    start_positions = []
    end_positions = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        sequence_ids = tokenized.sequence_ids(i)
        answer = examples['answers'][i]
        answer_start_char = answer['answer_start'][0]
        answer_text = answer['text'][0]
        answer_end_char = answer_start_char + len(answer_text)

        token_start = None
        token_end = None
        for idx, (offset_start, offset_end) in enumerate(offsets):
            if sequence_ids[idx] != 1:
                continue
            if offset_start >= answer_start_char and offset_end <= answer_end_char:
                if token_start is None:
                    token_start = idx
                token_end = idx
            elif offset_start < answer_end_char and offset_end > answer_start_char:
                if token_start is None:
                    token_start = idx
                token_end = idx

        if token_start is None or token_end is None:
            token_start = 0
            token_end = 0

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    return tokenized

# Evaluation

In [9]:
# Set up a trainer for evaluation
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
eval_args = TrainingArguments(
    output_dir=eval_dir,
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_drop_last=False,
    report_to=[],
)
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=eval_args,
)
trainer.label_names = ['start_positions', 'end_positions']

In [10]:
results = {}
squad_metric = evaluate.load('squad')

for i, lang in enumerate(LANGUAGES):
    print(f"\n{'=' * 64}")
    print(f"[{i+1}/{len(LANGUAGES)}] Evaluating: {lang}")
    print(f"{'=' * 64}")
    
    # Load and preprocess the test dataset for this language
    raw_dataset = load_test_dataset(lang, size=TEST_SIZE)
    test_dataset = raw_dataset.map(
        preprocess_squad, 
        batched=True, 
        remove_columns=raw_dataset.column_names
    )
    
    # Create a closure that captures this language's dataset
    def make_compute_metrics(raw_ds, test_ds):
        def compute_metrics_lang(pred):
            start_logits, end_logits = pred.predictions
            
            predictions = []
            for i, (start_log, end_log) in enumerate(zip(start_logits, end_logits)):
                start_idx = start_log.argmax()
                end_idx = end_log.argmax()
                input_ids = test_ds[i]['input_ids']
                answer_ids = input_ids[start_idx : end_idx + 1]
                answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
                predictions.append({
                    'id': raw_ds[i]['id'],
                    'prediction_text': answer_text,
                })
            
            references = []
            for example in raw_ds:
                references.append({
                    'id': example['id'],
                    'answers': {
                        'text': example['answers']['text'],
                        'answer_start': example['answers']['answer_start'],
                    },
                })
            
            return squad_metric.compute(predictions=predictions, references=references)
        return compute_metrics_lang
    
    # Update the trainer's compute_metrics for this language
    trainer.compute_metrics = make_compute_metrics(raw_dataset, test_dataset)
    
    # Run prediction
    predictions = trainer.predict(test_dataset)
    
    # Store results
    results[lang] = predictions.metrics
    
    print(f"Exact Match: {results[lang]['test_exact_match']:.2f}%")
    print(f"F1: {results[lang]['test_f1']:.2f}%")
    
    # Save per-language predictions
    pred_path = f"{eval_dir}/predictions_{lang}.txt"
    start_logits, end_logits = predictions.predictions
    with open(pred_path, 'w', encoding='utf-8') as f:
        for i in range(len(test_dataset)):
            start_idx = start_logits[i].argmax()
            end_idx = end_logits[i].argmax()
            input_ids = test_dataset[i]['input_ids']
            answer_ids = input_ids[start_idx:end_idx + 1]
            pred_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
            gt_text = raw_dataset[i]['answers']['text'][0]
            
            f.write(f"Q: {raw_dataset[i]['question']}\n")
            f.write(f"GT: {gt_text}\n")
            f.write(f"Pred: {pred_text}\n")
            f.write(f"Match: {pred_text.strip().lower() == gt_text.strip().lower()}\n")
            f.write("-" * 64 + "\n")
    
    print("-" * 64)
    print(f"Saved predictions to: {pred_path}")

# Print an overview table
print(f"\n{'=' * 64}")
print("Evaluation Result Overview")
print(f"{'=' * 64}")
print(f"{'Language':<10} {'Exact Match':>12} {'F1':>12}")
print("-" * 36)
for lang in LANGUAGES:
    print(f"{lang:<10} {results[lang]['test_exact_match']:>11.2f}% {results[lang]['test_f1']:>11.2f}%")

# Calculate averages
avg_em = sum(r['test_exact_match'] for r in results.values()) / len(results)
avg_f1 = sum(r['test_f1'] for r in results.values()) / len(results)

print("-" * 36)
print(f"{'Average':<10} {avg_em:>11.2f}% {avg_f1:>11.2f}%")


[1/11] Evaluating: en


Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Exact Match: 44.32%
F1: 55.22%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/predictions_en.txt

[2/11] Evaluating: ar


Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Exact Match: 28.48%
F1: 40.94%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/predictions_ar.txt

[3/11] Evaluating: de


...

Exact Match: 31.20%
F1: 42.54%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/predictions_tr.txt

[10/11] Evaluating: vi


Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Exact Match: 31.84%
F1: 46.51%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/predictions_vi.txt

[11/11] Evaluating: zh


Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Exact Match: 35.20%
F1: 43.72%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/predictions_zh.txt

Evaluation Result Overview
Language    Exact Match           F1
------------------------------------
en               44.32%       55.22%
ar               28.48%       40.94%
de               36.64%       48.37%
el               31.36%       41.65%
es               37.76%       48.27%
hi               32.64%       44.72%
ru               33.92%       46.57%
th               31.68%       40.74%
tr               31.20%       42.54%
vi               31.84%       46.51%
zh               35.20%       43.72%
------------------------------------
Average          34.09%       45.39%


In [11]:
# Save metrics JSON and CSV
metrics_json_path = f'{eval_dir}/metrics.json'
metrics_csv_path = f'{eval_dir}/metrics.csv'

metrics = []
for lang in LANGUAGES:
    metrics.append({
        'lang': lang,
        # 'exact_match': results[lang]['exact_match'],
        # 'f1': results[lang]['f1'],
        **results[lang],
    })
metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df[[
    'lang', 'test_loss', 'test_exact_match', 'test_f1', 
    'test_model_preparation_time', 'test_runtime', 
    'test_samples_per_second', 'test_steps_per_second'
]] # Rearrange columns

metrics_df.to_json(metrics_json_path, orient='records')
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"Saved metrics JSON to: {metrics_json_path}")
print(f"Saved metrics CSV to: {metrics_csv_path}")

Saved metrics JSON to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/metrics.json
Saved metrics CSV to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/metrics.csv


In [12]:
# Save metadata JSON
metadata_json_path = f'{eval_dir}/metadata.json'
metadata = {
    'model_id': MODEL_ID,
    'data_id': DATA_ID,
    'data_dir': DATA_DIR,
    'data_split': DATA_SPLIT,
    'test_size': TEST_SIZE,
    'batch_size': BATCH_SIZE,
    'evaluated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

with open(metadata_json_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)
    
print(f"Saved metadata JSON to: {metadata_json_path}")

Saved metadata JSON to: ./eval/xquad_xlmr/625/alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438/metadata.json


In [13]:
# Create a zip file containing the evaluation results
import shutil

# zip_dir = str(Path(eval_dir).parent.parent)
# zip_name = zip_dir.replace('/', '_').replace('\\', '_')
zip_dir = eval_dir
zip_name = eval_dir.rsplit('/', 1)[-1]

shutil.make_archive(zip_name, 'zip', zip_dir)

print(f"Created a zip file: {zip_name}.zip")

Created a zip file: XLM-R-Base-squad-en-5K-LoRA-Addition-v260719220438.zip
